In [1]:
# =============================================================================
# CELL 0: CONFIGURATION
# =============================================================================
from pathlib import Path
import os

# Data paths - resolve relative to notebook location
NOTEBOOK_DIR = Path(os.path.abspath(''))
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
DATA_FILE_PATTERN = 'bitcoin_lstm_features_v1.6_final.csv'  # Use v1.6 dataset with standard Lee-Mykland

# Train/val/test split ratios
TRAIN_SPLIT = 0.60
VAL_SPLIT = 0.20
TEST_SPLIT = 0.20

# Rolling normalization
DEFAULT_WINDOW = 720  # hours

# Window experiments
WINDOW_SIZES = [72, 168, 336, 720]  # hours

# Model hyperparameters
RANDOM_STATE = 42
LOGISTIC_MAX_ITER = 1000

# Tree model hyperparameters
RF_N_ESTIMATORS = 100
RF_MAX_DEPTH = 10
RF_MIN_SAMPLES_SPLIT = 10
RF_MIN_SAMPLES_LEAF = 4

XGB_N_ESTIMATORS = 100
XGB_MAX_DEPTH = 6
XGB_LEARNING_RATE = 0.1
XGB_SUBSAMPLE = 0.8
XGB_COLSAMPLE = 0.8

In [2]:
# =============================================================================
# CELL 1: IMPORTS AND DATA LOADING
# =============================================================================
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Load data - find data file dynamically
data_files = list(DATA_DIR.glob(DATA_FILE_PATTERN))
if not data_files:
    raise FileNotFoundError(f"No data files found matching {DATA_FILE_PATTERN} in {DATA_DIR}")
df = pd.read_csv(data_files[0])
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

In [3]:
# =============================================================================
# CELL 2: FEATURE DEFINITIONS
# =============================================================================

# Base features (no lags)
base_features = ['dvol', 'dvol_lag_1d', 'dvol_lag_7d', 'dvol_lag_30d', 
                 'network_activity', 'nvrv', 'dvol_rv_spread', 'transaction_volume']

# Jump features (using standard Lee-Mykland jump detection from v1.6)
jump_features = ['lee_mykland_jump', 'jump_magnitude', 'days_since_jump', 'jump_cluster_7d']

# Normalized feature sets (after rolling normalization)
market_features_norm = ['transaction_volume_norm', 'network_activity_norm', 
                        'nvrv_norm', 'dvol_rv_spread_norm']

core_features_norm = ['dvol_lag_1d_norm', 'dvol_lag_7d_norm', 'dvol_lag_30d_norm',
                      'transaction_volume_norm', 'network_activity_norm', 
                      'nvrv_norm', 'dvol_rv_spread_norm']

har_rv_features_norm = ['dvol_lag_1d_norm', 'dvol_lag_7d_norm', 'dvol_lag_30d_norm']

jump_feature_cols = ['lee_mykland_jump', 'jump_magnitude_norm', 'days_since_jump_norm', 'jump_cluster_7d_norm']

print(f"Features: {len(base_features)} base + {len(jump_features)} jump = {len(base_features)+len(jump_features)} total")

Features: 8 base + 4 jump = 12 total


In [4]:
# =============================================================================
# CELL 3: ROLLING NORMALIZATION
# =============================================================================

def apply_rolling_normalization(df, feature_cols, window=720):
    """Apply rolling window z-score normalization."""
    df_norm = df.copy()
    scaling_params = {}
    
    for col in feature_cols:
        # Don't normalize binary jump indicator
        if col in ['lee_mykland_jump']:
            df_norm[col] = df[col]
            continue
        rolling_mean = df[col].rolling(window=window, min_periods=1).mean()
        rolling_std = df[col].rolling(window=window, min_periods=1).std().replace(0, 1)
        df_norm[f'{col}_norm'] = (df[col] - rolling_mean) / rolling_std
        scaling_params[col] = {'mean': rolling_mean.iloc[-1], 'std': rolling_std.iloc[-1]}
    
    df_norm['dvol_rolling_mean'] = df['dvol'].rolling(window=window, min_periods=1).mean()
    df_norm['dvol_rolling_std'] = df['dvol'].rolling(window=window, min_periods=1).std().replace(0, 1)
    df_norm['timestamp'] = df['timestamp']
    return df_norm

# Apply rolling normalization
all_features = base_features + jump_features
df_norm = apply_rolling_normalization(df, all_features)

In [5]:
# =============================================================================
# CELL 4: TRAIN/VAL/TEST SPLIT
# =============================================================================

n_train = int(len(df_norm) * TRAIN_SPLIT)
n_val = int(len(df_norm) * VAL_SPLIT)

train_df = df_norm.iloc[:n_train].copy()
val_df = df_norm.iloc[n_train:n_train + n_val].copy()
test_df = df_norm.iloc[n_train + n_val:].copy()

# Create binary target for ALL dataframes
train_df['direction_binary'] = (df['dvol'].shift(-1).iloc[:len(train_df)] > df['dvol'].iloc[:len(train_df)]).astype(int)
val_df['direction_binary'] = (df['dvol'].shift(-1).iloc[n_train:n_train + n_val] > df['dvol'].iloc[n_train:n_train + n_val]).astype(int).values
test_df['direction_binary'] = (df['dvol'].shift(-1).iloc[n_train + n_val:] > df['dvol'].iloc[n_train + n_val:]).astype(int).values

# Calculate baselines from data (not hardcoded)
baseline_random = 0.5  # Theoretical baseline for binary classification
y_test_actual = df['dvol'].shift(-1).iloc[n_train + n_val + 1:].dropna()
baseline_majority = max(y_test_actual.mean(), 1 - y_test_actual.mean())

print(f"Samples: {len(train_df):,} train | {len(val_df):,} val | {len(test_df):,} test")
print(f"Baselines: random={baseline_random:.4f}, majority={baseline_majority:.4f}")

Samples: 24,633 train | 8,211 val | 8,211 test
Baselines: random=0.5000, majority=45.1893


In [6]:
# =============================================================================
# CELL 5: HELPER FUNCTIONS
# =============================================================================
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, confusion_matrix)
import scipy.stats as stats

def prepare_classification_data(train_df, val_df, test_df, feature_cols):
    """Prepare train/val/test splits for classification."""
    # Binary target (already created)
    y_train = train_df['direction_binary'].shift(-1)
    y_val = val_df['direction_binary'].shift(-1)
    y_test = test_df['direction_binary'].shift(-1)
    
    # Features
    X_train = train_df[feature_cols]
    X_val = val_df[feature_cols]
    X_test = test_df[feature_cols]
    
    # Remove NaN rows
    valid_train = (~y_train.isna()) & (~X_train.isna().any(axis=1))
    valid_val = (~y_val.isna()) & (~X_val.isna().any(axis=1))
    valid_test = (~y_test.isna()) & (~X_test.isna().any(axis=1))
    
    y_train = y_train[valid_train].astype(int)
    y_val = y_val[valid_val].astype(int)
    y_test = y_test[valid_test].astype(int)
    
    X_train = X_train[valid_train].reset_index(drop=True)
    X_val = X_val[valid_val].reset_index(drop=True)
    X_test = X_test[valid_test].reset_index(drop=True)
    
    return X_train, X_val, X_test, y_train, y_val, y_test

def pesaran_timmermann_test(y_true, y_pred):
    """
    Pesaran-Timmermann test for directional accuracy significance.

    Tests if directional accuracy is significantly better than random guessing.
    Uses the full standard formula from Pesaran & Timmermann (1992).

    Args:
        y_true: Actual binary labels (0 or 1)
        y_pred: Predicted probabilities or binary labels

    Returns:
        (pt_stat, p_value): Test statistic and two-tailed p-value

    Reference: Pesaran, M. H., & Timmermann, A. (1992).
    "A simple nonparametric test of predictive performance".
    Journal of Business & Economic Statistics, 10(4), 461-465.
    """
    n = len(y_true)
    y_pred_binary = (y_pred > 0.5).astype(int) if y_pred.dtype == float else y_pred

    # Observed directional accuracy
    p_hat = np.mean(y_true == y_pred_binary)

    # Proportion of up movements in actuals
    P_y = np.mean(y_true)

    # Proportion of up predictions
    P_f = np.mean(y_pred_binary)

    # Expected accuracy under null (no predictive power)
    # This is the FULL standard formula from Pesaran & Timmermann (1992)
    P_ye = P_y * P_f + (1 - P_y) * (1 - P_f)

    # Variance under null
    var_pt = P_ye * (1 - P_ye) / n

    if var_pt == 0:
        return np.nan, np.nan

    # Test statistic
    pt_stat = (p_hat - P_ye) / np.sqrt(var_pt)

    # Two-tailed p-value
    p_value = 2 * (1 - stats.norm.cdf(abs(pt_stat)))

    return pt_stat, p_value

def evaluate_classification(model, X_train, y_train, X_val, y_val, X_test, y_test, needs_proba=False):
    """
    Evaluate classification model on train/val/test splits.

    Args:
        model: Trained classification model
        X_train, y_train: Training data
        X_val, y_val: Validation data
        X_test, y_test: Test data
        needs_proba: Whether to use predict_proba

    Returns:
        Dictionary with metrics for train/val/test splits
    """
    results = {}
    
    for name, X, y_true in [('train', X_train, y_train), 
                            ('val', X_val, y_val), 
                            ('test', X_test, y_test)]:
        if needs_proba:
            y_proba = model.predict_proba(X)[:, 1]
            y_pred = (y_proba > 0.5).astype(int)
        else:
            y_pred = model.predict(X)
            y_proba = y_pred
        
        accuracy = accuracy_score(y_true, y_pred)
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        
        if needs_proba:
            try:
                roc_auc = roc_auc_score(y_true, y_proba)
            except:
                roc_auc = np.nan
        else:
            roc_auc = np.nan
        
        # PT test
        # Handle both pandas Series and numpy arrays
        y_true_array = y_true.values if hasattr(y_true, "values") else y_true
        pt_stat, p_value = pesaran_timmermann_test(y_true_array, y_pred)
        
        results[name] = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'roc_auc': roc_auc,
            'pt_stat': pt_stat,
            'pt_pvalue': p_value
        }
    
    return results

In [7]:
# =============================================================================
# CELL 6: LINEAR CLASSIFICATION MODELS
# =============================================================================
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

linear_classification_results = {}

# Model specifications - ALL LINEAR MODELS
linear_specs = [
    # No Lags, No Jumps (4 features)
    ('Logistic_NoLags', market_features_norm, 'Logistic'),
    ('LDA_NoLags', market_features_norm, 'LDA'),
    # HAR-RV only (3 features)
    ('Logistic_HAR', har_rv_features_norm, 'Logistic'),
    ('LDA_HAR', har_rv_features_norm, 'LDA'),
    # With Lags, No Jumps (7 features)
    ('Logistic_WithLags', core_features_norm, 'Logistic'),
    ('LDA_WithLags', core_features_norm, 'LDA'),
    # No Lags, With Jumps (8 features)
    ('Logistic_NoLags_Jumps', market_features_norm + jump_feature_cols, 'Logistic'),
    ('LDA_NoLags_Jumps', market_features_norm + jump_feature_cols, 'LDA'),
    # With Lags, With Jumps (11 features)
    ('Logistic_WithLags_Jumps', core_features_norm + jump_feature_cols, 'Logistic'),
    ('LDA_WithLags_Jumps', core_features_norm + jump_feature_cols, 'LDA'),
]

for name, features, model_type in linear_specs:
    X_train, X_val, X_test, y_train, y_val, y_test = prepare_classification_data(
        train_df, val_df, test_df, features)
    
    if model_type == 'Logistic':
        model = LogisticRegression(class_weight='balanced', random_state=RANDOM_STATE, max_iter=LOGISTIC_MAX_ITER)
        needs_proba = True
    else:  # LDA
        model = LDA()
        needs_proba = False
    
    model.fit(X_train, y_train)
    metrics = evaluate_classification(model, X_train, y_train, X_val, y_val, X_test, y_test, needs_proba=needs_proba)
    linear_classification_results[name] = {'model': model, 'features': features, 'metrics': metrics}
    
    m = metrics['test']
    sig = '***' if m['pt_pvalue'] < 0.01 else ('**' if m['pt_pvalue'] < 0.05 else ('*' if m['pt_pvalue'] < 0.1 else ''))
    print(f"{name:25s} Acc={m['accuracy']:.4f} F1={m['f1']:.4f} AUC={m['roc_auc']:.4f} PT={m['pt_stat']:.2f}{sig}")

Logistic_NoLags           Acc=0.5029 F1=0.4878 AUC=0.5054 PT=0.74
LDA_NoLags                Acc=0.5397 F1=0.0857 AUC=nan PT=0.11
Logistic_HAR              Acc=0.4916 F1=0.4806 AUC=0.4951 PT=-1.18
LDA_HAR                   Acc=0.5429 F1=0.0000 AUC=nan PT=-0.04
Logistic_WithLags         Acc=0.4998 F1=0.4745 AUC=0.5033 PT=-0.12
LDA_WithLags              Acc=0.5397 F1=0.1331 AUC=nan PT=0.54
Logistic_NoLags_Jumps     Acc=0.5006 F1=0.4841 AUC=0.5052 PT=0.29
LDA_NoLags_Jumps          Acc=0.5379 F1=0.1009 AUC=nan PT=-0.06


Logistic_WithLags_Jumps   Acc=0.5028 F1=0.4771 AUC=0.5020 PT=0.41
LDA_WithLags_Jumps        Acc=0.5384 F1=0.1359 AUC=nan PT=0.35


In [8]:
# =============================================================================
# CELL 7: TREE-BASED CLASSIFICATION MODELS
# =============================================================================
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

tree_classification_results = {}

# Helper for adding jump features - FIXED to handle filtered data with reset index
def prepare_jump_features(X_base, df_source, jump_cols):
    # X_base has reset index (0, 1, 2, ...) after prepare_classification_data
    # We need to use the original positional indices from df_source
    # Since prepare_classification_data filters and resets, we use positional indexing
    n_rows = len(X_base)
    jump_feats = df_source[jump_cols].iloc[:n_rows].reset_index(drop=True)
    return pd.concat([X_base.reset_index(drop=True), jump_feats], axis=1)

# Prepare data for each model type - store all needed data
# NOTE: prepare_classification_data returns X with RESET index
rf_nolag_data = prepare_classification_data(train_df, val_df, test_df, market_features_norm)
X_train_rf_nolag, X_val_rf_nolag, X_test_rf_nolag, y_train_rf, y_val_rf, y_test_rf = rf_nolag_data

rf_lags_data = prepare_classification_data(train_df, val_df, test_df, core_features_norm)
X_train_rf_lags, X_val_rf_lags, X_test_rf_lags, y_train_rf_lags, y_val_rf_lags, y_test_rf_lags = rf_lags_data

# Add jump features to FILTERED data
# Use iloc positional indexing since both X_base and df_source are aligned positionally
X_train_rf_nolag_jumps = prepare_jump_features(X_train_rf_nolag.copy(), train_df, jump_feature_cols)
X_val_rf_nolag_jumps = prepare_jump_features(X_val_rf_nolag.copy(), val_df, jump_feature_cols)
X_test_rf_nolag_jumps = prepare_jump_features(X_test_rf_nolag.copy(), test_df, jump_feature_cols)

X_train_rf_lags_jumps = prepare_jump_features(X_train_rf_lags.copy(), train_df, jump_feature_cols)
X_val_rf_lags_jumps = prepare_jump_features(X_val_rf_lags.copy(), val_df, jump_feature_cols)
X_test_rf_lags_jumps = prepare_jump_features(X_test_rf_lags.copy(), test_df, jump_feature_cols)

# Random Forest models - use correct y for each feature set
rf_models = [
    ('RF_NoLag', X_train_rf_nolag, X_val_rf_nolag, X_test_rf_nolag, y_train_rf, y_val_rf, y_test_rf, market_features_norm),
    ('RF_Lags', X_train_rf_lags, X_val_rf_lags, X_test_rf_lags, y_train_rf_lags, y_val_rf_lags, y_test_rf_lags, core_features_norm),
    ('RF_NoLag_Jumps', X_train_rf_nolag_jumps, X_val_rf_nolag_jumps, X_test_rf_nolag_jumps, y_train_rf, y_val_rf, y_test_rf, market_features_norm + jump_feature_cols),
    ('RF_Lags_Jumps', X_train_rf_lags_jumps, X_val_rf_lags_jumps, X_test_rf_lags_jumps, y_train_rf_lags, y_val_rf, y_test_rf_lags, core_features_norm + jump_feature_cols),
]

for name, X_tr, X_v, X_te, y_tr, y_va, y_te, features in rf_models:
    model = RandomForestClassifier(
        n_estimators=RF_N_ESTIMATORS, max_depth=RF_MAX_DEPTH, min_samples_split=RF_MIN_SAMPLES_SPLIT,
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
    )
    model.fit(X_tr, y_tr)
    metrics = evaluate_classification(model, X_tr, y_tr, X_v, y_va, X_te, y_te, needs_proba=True)
    tree_classification_results[name] = {'model': model, 'features': features, 'metrics': metrics}
    print(f"{name}: Test Acc = {metrics['test']['accuracy']:.4f}")

# XGBoost models - use correct y for each feature set
xgb_models = [
    ('XGB_NoLag', X_train_rf_nolag, X_val_rf_nolag, X_test_rf_nolag, y_train_rf, y_val_rf, y_test_rf, market_features_norm),
    ('XGB_Lags', X_train_rf_lags, X_val_rf_lags, X_test_rf_lags, y_train_rf_lags, y_val_rf_lags, y_test_rf_lags, core_features_norm),
    ('XGB_NoLag_Jumps', X_train_rf_nolag_jumps, X_val_rf_nolag_jumps, X_test_rf_nolag_jumps, y_train_rf, y_val_rf, y_test_rf, market_features_norm + jump_feature_cols),
    ('XGB_Lags_Jumps', X_train_rf_lags_jumps, X_val_rf_lags_jumps, X_test_rf_lags_jumps, y_train_rf_lags, y_val_rf, y_test_rf_lags, core_features_norm + jump_feature_cols),
]

for name, X_tr, X_v, X_te, y_tr, y_va, y_te, features in xgb_models:
    model = XGBClassifier(
        n_estimators=XGB_N_ESTIMATORS, max_depth=XGB_MAX_DEPTH, learning_rate=XGB_LEARNING_RATE,
        subsample=XGB_SUBSAMPLE, colsample_bytree=XGB_COLSAMPLE,
        random_state=RANDOM_STATE, n_jobs=-1, eval_metric='logloss'
    )
    model.fit(X_tr, y_tr)
    metrics = evaluate_classification(model, X_tr, y_tr, X_v, y_va, X_te, y_te, needs_proba=True)
    tree_classification_results[name] = {'model': model, 'features': features, 'metrics': metrics}
    print(f"{name}: Test Acc = {metrics['test']['accuracy']:.4f}")

RF_NoLag: Test Acc = 0.5139


RF_Lags: Test Acc = 0.5067


RF_NoLag_Jumps: Test Acc = 0.5071


RF_Lags_Jumps: Test Acc = 0.5079


XGB_NoLag: Test Acc = 0.5296
XGB_Lags: Test Acc = 0.5219


XGB_NoLag_Jumps: Test Acc = 0.5270


XGB_Lags_Jumps: Test Acc = 0.5164


In [9]:
# =============================================================================
# CELL 8: COMBINED SUMMARY TABLE
# =============================================================================
print("\n" + "="*100)
print("TEST SET PERFORMANCE (ALL CLASSIFICATION MODELS)")
print("="*100)
print(f"{'Model':<25} {'Type':<10} {'Feats':>5} {'Accuracy':>10} {'Precision':>9} {'Recall':>7} {'F1':>6} {'AUC':>6}")
print("-"*100)

# Linear models
for name in linear_classification_results.keys():
    m = linear_classification_results[name]['metrics']['test']
    f = len(linear_classification_results[name]['features'])
    print(f"{name:<25} {'Linear':<10} {f:>5} {m['accuracy']:>9.4f} {m['precision']:>9.4f} {m['recall']:>7.4f} {m['f1']:>6.4f} {m['roc_auc']:>6.4f}")

# Tree models
for name in tree_classification_results.keys():
    m = tree_classification_results[name]['metrics']['test']
    f = len(tree_classification_results[name]['features'])
    print(f"{name:<25} {'Tree':<10} {f:>5} {m['accuracy']:>9.4f} {m['precision']:>9.4f} {m['recall']:>7.4f} {m['f1']:>6.4f} {m['roc_auc']:>6.4f}")

print("-"*100)
print(f"\nBaselines:")
print(f"  Random guess:           {baseline_random:.4f}")
print(f"  Always majority class:  {baseline_majority:.4f}")


TEST SET PERFORMANCE (ALL CLASSIFICATION MODELS)
Model                     Type       Feats   Accuracy Precision  Recall     F1    AUC
----------------------------------------------------------------------------------------------------
Logistic_NoLags           Linear         4    0.5029    0.4609  0.5180 0.4878 0.5054
LDA_NoLags                Linear         4    0.5397    0.4634  0.0472 0.0857    nan
Logistic_HAR              Linear         3    0.4916    0.4506  0.5148 0.4806 0.4951
LDA_HAR                   Linear         3    0.5429    0.0000  0.0000 0.0000    nan
Logistic_WithLags         Linear         7    0.4998    0.4562  0.4943 0.4745 0.5033
LDA_WithLags              Linear         7    0.5397    0.4770  0.0773 0.1331    nan
Logistic_NoLags_Jumps     Linear         8    0.5006    0.4584  0.5129 0.4841 0.5052
LDA_NoLags_Jumps          Linear         8    0.5379    0.4542  0.0568 0.1009    nan
Logistic_WithLags_Jumps   Linear        11    0.5028    0.4592  0.4964 0.4771 0.502

In [10]:
# =============================================================================
# CELL 9: PESARAN-TIMMERMANN SIGNIFICANCE TEST SUMMARY
# =============================================================================
print("\n" + "="*90)
print("PESARAN-TIMMERMANN SIGNIFICANCE TEST (Directional Accuracy vs Random)")
print("="*90)
print(f"{'Model':<25} {'Accuracy':>10} {'PT-stat':>9} {'p-value':>10} {'Significant':>12}")
print("-"*90)

all_results = {**linear_classification_results, **tree_classification_results}

for name in sorted(all_results.keys()):
    m = all_results[name]['metrics']['test']
    sig = '***' if m['pt_pvalue'] < 0.01 else ('**' if m['pt_pvalue'] < 0.05 else ('*' if m['pt_pvalue'] < 0.1 else ''))
    print(f"{name:<25} {m['accuracy']:>9.4f} {m['pt_stat']:>9.2f} {m['pt_pvalue']:>10.4f} {sig:>12}")

print("\nNote: *** p<0.01, ** p<0.05, * p<0.1")
print("PT-stat > 1.96 indicates directional accuracy significantly better than random at 5% level")


PESARAN-TIMMERMANN SIGNIFICANCE TEST (Directional Accuracy vs Random)
Model                       Accuracy   PT-stat    p-value  Significant
------------------------------------------------------------------------------------------
LDA_HAR                      0.5429     -0.04     0.9677             
LDA_NoLags                   0.5397      0.11     0.9129             
LDA_NoLags_Jumps             0.5379     -0.06     0.9549             
LDA_WithLags                 0.5397      0.54     0.5887             
LDA_WithLags_Jumps           0.5384      0.35     0.7272             
Logistic_HAR                 0.4916     -1.18     0.2378             
Logistic_NoLags              0.5029      0.74     0.4587             
Logistic_NoLags_Jumps        0.5006      0.29     0.7753             
Logistic_WithLags            0.4998     -0.12     0.9027             
Logistic_WithLags_Jumps      0.5028      0.41     0.6800             
RF_Lags                      0.5067     -0.03     0.9784           

In [11]:
# =============================================================================
# CELL 10: SAVE RESULTS TO JSON
# =============================================================================
import json
from datetime import datetime

# Prepare results for saving
results_summary = {
    'date': datetime.now().strftime('%Y-%m-%d'),
    'n_samples': {
        'train': len(train_df),
        'val': len(val_df),
        'test': len(test_df)
    },
    'baselines': {
        'random_guess': baseline_random,
        'always_majority': baseline_majority
    },
    'models': {}
}

for name, result in all_results.items():
    results_summary['models'][name] = {
        'n_features': len(result['features']),
        'test_metrics': {
            'accuracy': float(result['metrics']['test']['accuracy']),
            'precision': float(result['metrics']['test']['precision']),
            'recall': float(result['metrics']['test']['recall']),
            'f1': float(result['metrics']['test']['f1']),
            'roc_auc': float(result['metrics']['test']['roc_auc']),
            'pt_stat': float(result['metrics']['test']['pt_stat']),
            'pt_pvalue': float(result['metrics']['test']['pt_pvalue'])
        },
        'val_metrics': {
            'accuracy': float(result['metrics']['val']['accuracy']),
            'f1': float(result['metrics']['val']['f1']),
            'roc_auc': float(result['metrics']['val']['roc_auc'])
        }
    }

# Find best model
best_model_name = max(all_results.keys(), 
                      key=lambda n: all_results[n]['metrics']['test']['accuracy'])
results_summary['best_model'] = {
    'name': best_model_name,
    'test_accuracy': float(all_results[best_model_name]['metrics']['test']['accuracy']),
    'test_f1': float(all_results[best_model_name]['metrics']['test']['f1']),
    'test_roc_auc': float(all_results[best_model_name]['metrics']['test']['roc_auc'])
}

# Save results
output_dir = Path('results/analysis')
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'classification_results.json'

with open(output_path, 'w') as f:
    json.dump(results_summary, f, indent=2)

print(f"\nResults saved to: {output_path}")
print(f"\nBest Model: {best_model_name}")
print(f"  Test Accuracy: {results_summary['best_model']['test_accuracy']:.4f}")
print(f"  Test F1: {results_summary['best_model']['test_f1']:.4f}")
print(f"  Test AUC: {results_summary['best_model']['test_roc_auc']:.4f}")
print(f"\nBaseline Comparison:")
print(f"  Random guess: {baseline_random:.4f}")
print(f"  Always majority: {baseline_majority:.4f}")
print(f"\nImprovement over random: {(results_summary['best_model']['test_accuracy'] - baseline_random)*100:.2f} percentage points")


Results saved to: results/analysis/classification_results.json

Best Model: LDA_HAR
  Test Accuracy: 0.5429
  Test F1: 0.0000
  Test AUC: nan

Baseline Comparison:
  Random guess: 0.5000
  Always majority: 45.1893

Improvement over random: 4.29 percentage points


In [12]:
# =============================================================================
# CELL 11: MULTI-WINDOW EXPERIMENTS
# =============================================================================
# Per Chung et al. (2025), different window sizes adapt to different
# regime dynamics.

def run_experiments_for_window(window_size):
    """Run all classification models for a specific window size."""
    
    # Apply rolling normalization with specified window
    df_norm_w = apply_rolling_normalization(df, all_features, window=window_size)
    
    # Create binary target
    df_norm_w['direction_binary'] = (df['dvol'].shift(-1) > df['dvol']).astype(int)
    
    # Train/val/test split (using config constants)
    n_train = int(len(df_norm_w) * TRAIN_SPLIT)
    n_val = int(len(df_norm_w) * VAL_SPLIT)
    
    train_df_w = df_norm_w.iloc[:n_train].copy()
    val_df_w = df_norm_w.iloc[n_train:n_train + n_val].copy()
    test_df_w = df_norm_w.iloc[n_train + n_val:].copy()
    
    # Re-define normalized feature columns for window experiment
    market_norm = ['transaction_volume_norm', 'network_activity_norm', 
                   'nvrv_norm', 'dvol_rv_spread_norm']
    core_norm = ['dvol_lag_1d_norm', 'dvol_lag_7d_norm', 'dvol_lag_30d_norm',
                 'transaction_volume_norm', 'network_activity_norm', 
                 'nvrv_norm', 'dvol_rv_spread_norm']
    har_norm = ['dvol_lag_1d_norm', 'dvol_lag_7d_norm', 'dvol_lag_30d_norm']
    jump_cols = ['lee_mykland_jump', 'jump_magnitude_norm', 'days_since_jump_norm', 'jump_cluster_7d_norm']
    
    # Model specifications - ALL 18 MODELS (matching main experiments)
    model_specs = [
        # No Lags, No Jumps (4 features)
        ('Logistic_NoLags', market_norm, 'Logistic'),
        ('LDA_NoLags', market_norm, 'LDA'),
        ('RF_NoLag', market_norm, 'RF'),
        ('XGB_NoLag', market_norm, 'XGB'),
        # HAR-RV only (3 features)
        ('Logistic_HAR', har_norm, 'Logistic'),
        ('LDA_HAR', har_norm, 'LDA'),
        # With Lags, No Jumps (7 features)
        ('Logistic_WithLags', core_norm, 'Logistic'),
        ('LDA_WithLags', core_norm, 'LDA'),
        ('RF_Lags', core_norm, 'RF'),
        ('XGB_Lags', core_norm, 'XGB'),
        # No Lags, With Jumps (8 features)
        ('Logistic_NoLags_Jumps', market_norm + jump_cols, 'Logistic'),
        ('LDA_NoLags_Jumps', market_norm + jump_cols, 'LDA'),
        ('RF_NoLag_Jumps', market_norm + jump_cols, 'RF'),
        ('XGB_NoLag_Jumps', market_norm + jump_cols, 'XGB'),
        # With Lags, With Jumps (11 features)
        ('Logistic_WithLags_Jumps', core_norm + jump_cols, 'Logistic'),
        ('LDA_WithLags_Jumps', core_norm + jump_cols, 'LDA'),
        ('RF_Lags_Jumps', core_norm + jump_cols, 'RF'),
        ('XGB_Lags_Jumps', core_norm + jump_cols, 'XGB'),
    ]
    
    window_results = {}
    
    for name, features, model_type in model_specs:
        try:
            # Prepare data
            # Extract features as numpy arrays
            X_train_w = train_df_w[features].values
            X_val_w = val_df_w[features].values
            X_test_w = test_df_w[features].values
            
            # Extract target variable
            y_train_w = train_df_w['direction_binary'].values
            y_val_w = val_df_w['direction_binary'].values
            y_test_w = test_df_w['direction_binary'].values
            
            # Combine X and y to filter rows with any NaN
            train_mask = ~np.isnan(X_train_w).any(axis=1) & ~np.isnan(y_train_w)
            val_mask = ~np.isnan(X_val_w).any(axis=1) & ~np.isnan(y_val_w)
            test_mask = ~np.isnan(X_test_w).any(axis=1) & ~np.isnan(y_test_w)
            
            X_train_w = X_train_w[train_mask]
            y_train_w = y_train_w[train_mask]
            X_val_w = X_val_w[val_mask]
            y_val_w = y_val_w[val_mask]
            X_test_w = X_test_w[test_mask]
            y_test_w = y_test_w[test_mask]
            
            # Initialize and train model based on type
            if model_type == 'Logistic':
                model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
            elif model_type == 'LDA':
                model = LDA()
            elif model_type == 'RF':
                model = RandomForestClassifier(
                    n_estimators=100, max_depth=10, min_samples_split=10,
                    min_samples_leaf=4, random_state=RANDOM_STATE, n_jobs=-1)
            elif model_type == 'XGB':
                model = XGBClassifier(
                    n_estimators=100, max_depth=6, learning_rate=0.1,
                    subsample=0.8, colsample_bytree=0.8, 
                    random_state=RANDOM_STATE, n_jobs=-1, eval_metric='logloss')
            
            # Train
            model.fit(X_train_w, y_train_w)
            
            # Evaluate
            metrics = evaluate_classification(
                model, X_train_w, y_train_w, X_val_w, y_val_w, 
                X_test_w, y_test_w, needs_proba=(model_type in ['Logistic', 'XGB'])
            )
            
            window_results[name] = {'model': model, 'features': features, 'metrics': metrics}
            
        except Exception as e:
            print(f"  Warning: {name} failed: {e}")
            window_results[name] = None
    
    return window_results

# Run experiments for all window sizes
all_window_results = {}
for window in WINDOW_SIZES:
    all_window_results[window] = run_experiments_for_window(window)

# Compile results for comparison
window_comparison = []

for window in WINDOW_SIZES:
    results = all_window_results[window]
    for name, result in results.items():
        if result is not None:
            window_comparison.append({
                'window': window,
                'window_days': window // 24,
                'model': name,
                'accuracy': float(result['metrics']['test']['accuracy']),
                'f1': float(result['metrics']['test']['f1']),
                'roc_auc': float(result['metrics']['test']['roc_auc'])
            })

# Create DataFrame for analysis
window_df = pd.DataFrame(window_comparison)

# Find best model for each window
print("\n" + "="*90)
print("BEST MODEL BY WINDOW SIZE")
print("="*90)

if not window_df.empty:
    for window in WINDOW_SIZES:
        window_data = window_df[window_df['window'] == window]
        if not window_data.empty:
            best = window_data.loc[window_data['accuracy'].idxmax()]
            print(f"Window {window}h ({window//24}d): {best['model']} | Acc={best['accuracy']:.4f} | F1={best['f1']:.4f}")

    # Overall best across all windows
    overall_best = window_df.loc[window_df['accuracy'].idxmax()]
    print("\n" + "-"*90)
    print(f"OVERALL BEST: {overall_best['model']} at {overall_best['window']}h window")
    print(f"  Accuracy: {overall_best['accuracy']:.4f}")
    print(f"  F1: {overall_best['f1']:.4f}")
    print(f"  AUC: {overall_best['roc_auc']:.4f}")
else:
    print("\n" + "-"*90)
    print("No results to display - all models failed to train")
    print("Please check the error messages above for details")


BEST MODEL BY WINDOW SIZE
Window 72h (3d): LDA_NoLags_Jumps | Acc=0.5462 | F1=0.0886
Window 168h (7d): Logistic_NoLags | Acc=0.5443 | F1=0.0756
Window 336h (14d): Logistic_NoLags | Acc=0.5435 | F1=0.0841
Window 720h (30d): LDA_NoLags | Acc=0.5443 | F1=0.0783

------------------------------------------------------------------------------------------
OVERALL BEST: LDA_NoLags_Jumps at 72h window
  Accuracy: 0.5462
  F1: 0.0886
  AUC: nan


In [13]:
# =============================================================================
# CELL 12: WINDOW COMPARISON TABLE
# =============================================================================

print("\n" + "="*100)
print("WINDOW SIZE COMPARISON - ACCURACY BY MODEL AND WINDOW")
print("="*100)
print(f"{'Model':<20} {'72h':<8} {'168h':<8} {'336h':<8} {'720h':<8} {'Best':<8} {'Range':<8}")
print("-"*100)

for model_name in all_window_results[72].keys():
    # Get accuracies, use NaN for failed models
    accuracies = [all_window_results[w][model_name]['metrics']['test']['accuracy'] if all_window_results[w][model_name] is not None else np.nan for w in WINDOW_SIZES]
    # Skip if all NaN
    if all(np.isnan(accuracies)):
        continue
    best_window = WINDOW_SIZES[np.nanargmax(accuracies)]
    best_acc = np.nanmax(accuracies)
    acc_range = np.nanmax(accuracies) - np.nanmin(accuracies)
    print(f"{model_name:<20} {accuracies[0]:>7.4f} {accuracies[1]:>7.4f} {accuracies[2]:>7.4f} {accuracies[3]:>7.4f} {best_window:>7}h {acc_range:>7.4f}")

print("-"*100)

# Calculate average accuracy per window
avg_acc_per_window = {w: np.mean([all_window_results[w][m]['metrics']['test']['accuracy'] for m in all_window_results[w].keys() if all_window_results[w][m] is not None]) for w in WINDOW_SIZES}
print(f"{'AVERAGE':<20} {avg_acc_per_window[72]:>7.4f} {avg_acc_per_window[168]:>7.4f} {avg_acc_per_window[336]:>7.4f} {avg_acc_per_window[720]:>7.4f}")
print(f"\nBest average window: {max(avg_acc_per_window, key=avg_acc_per_window.get)}h ({max(avg_acc_per_window.values()):.4f})")


WINDOW SIZE COMPARISON - ACCURACY BY MODEL AND WINDOW
Model                72h      168h     336h     720h     Best     Range   
----------------------------------------------------------------------------------------------------
Logistic_NoLags       0.5423  0.5443  0.5435  0.5441     168h  0.0019
LDA_NoLags            0.5424  0.5443  0.5435  0.5443     168h  0.0018
RF_NoLag              0.5396  0.5376  0.5422  0.5356     336h  0.0066
XGB_NoLag             0.5269  0.5290  0.5310  0.5284     336h  0.0041
Logistic_HAR          0.5431  0.5429  0.5433  0.5431     336h  0.0004
LDA_HAR               0.5431  0.5429  0.5432  0.5431     336h  0.0002
Logistic_WithLags     0.5427  0.5411  0.5416  0.5433     720h  0.0022
LDA_WithLags          0.5426  0.5412  0.5416  0.5435     720h  0.0023
RF_Lags               0.5377  0.5282  0.5365  0.5328      72h  0.0095
XGB_Lags              0.5271  0.5264  0.5185  0.5275     720h  0.0090
Logistic_NoLags_Jumps  0.5456  0.5439  0.5435  0.5437      72h  0.002

In [14]:
# =============================================================================
# CELL 13: SAVE WINDOW RESULTS
# =============================================================================

window_summary = {
    'date': datetime.now().strftime('%Y-%m-%d'),
    'window_sizes_tested': WINDOW_SIZES,
    'baseline_random': baseline_random,
    'results_by_window': {},
    'best_window_per_model': {},
    'summary_statistics': {}
}

# Results by window
for window in WINDOW_SIZES:
    window_results = {}
    for model_name, result in all_window_results[window].items():
        if result is not None:
            metrics = result['metrics']['test']
            window_results[model_name] = {
                'accuracy': float(metrics['accuracy']),
                'f1': float(metrics['f1']),
                'roc_auc': float(metrics['roc_auc']) if not np.isnan(metrics['roc_auc']) else None,
                'pt_stat': float(metrics['pt_stat']) if not np.isnan(metrics['pt_stat']) else None,
                'pt_pvalue': float(metrics['pt_pvalue']) if not np.isnan(metrics['pt_pvalue']) else None
            }
    window_summary['results_by_window'][f'window_{window}h'] = window_results

# Best window per model
model_names = set()
for window in WINDOW_SIZES:
    for model_name in all_window_results[window].keys():
        model_names.add(model_name)

for model_name in model_names:
    accuracies = []
    for w in WINDOW_SIZES:
        result = all_window_results[w][model_name]
        if result is not None:
            accuracies.append((w, result['metrics']['test']['accuracy']))
    if accuracies:
        best_window, best_acc = max(accuracies, key=lambda x: x[1])
        window_summary['best_window_per_model'][model_name] = {
            'best_window_hours': best_window,
            'best_window_days': best_window // 24,
            'best_accuracy': float(best_acc)
        }

# Summary statistics
window_summary['summary_statistics'] = {
    'average_accuracy_by_window': {
        f'{w}h': float(avg_acc_per_window[w]) for w in WINDOW_SIZES
    },
    'best_overall_window': {
        'hours': int(max(avg_acc_per_window, key=avg_acc_per_window.get)),
        'days': max(avg_acc_per_window, key=avg_acc_per_window.get) // 24,
        'average_accuracy': float(max(avg_acc_per_window.values()))
    },
    'improvement_over_baseline': {
        f'{w}h': float(avg_acc_per_window[w] - baseline_random) for w in WINDOW_SIZES
    }
}

# Save results
window_output_path = output_dir / 'classification_window_comparison_results.json'
with open(window_output_path, 'w') as f:
    json.dump(window_summary, f, indent=2)

print(f"\nWindow results saved to: {window_output_path}")


Window results saved to: results/analysis/classification_window_comparison_results.json
